In [ ]:

import os
import glob
import shutil
import zipfile
import warnings
import calendar
import urllib.request
import requests
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
warnings.filterwarnings('ignore')

In [ ]:

# Format: "Region_Name": (lat_min, lat_max, lon_min, lon_max)
TARGET_REGIONS = {
    "Arabian_Sea": (5, 20, 50, 100), 
    "Bay_of_Bengal": (5, 20, 80, 100),
    "Equatorial_IO": (-5, 5, 50, 100)
}

BASE_URL = "https://data-argo.ifremer.fr/dac"

def download_bioargo_data(index_df):
    """
    Downloads regional BGC-Argo files by filtering a master index dataset.

    Inputs:
    -------
    index_df : pandas.DataFrame
        The master Argo index database containing metadata for global profiles.
        Required file format: In-memory pandas DataFrame (parsed from a .txt or .csv index file).
        Required columns/schema:
        - 'latitude' : float, geographic coordinates (-90 to 90).
        - 'longitude' : float, geographic coordinates (-180 to 180).
        - 'parameters' : str, space-separated listing of variables (must contain 'CHLA').
        - 'file' : str, server path string structured as 'dac/wmo/file_name.nc'.

    What it does:
    -------------
    1. Iterates through defined geographic boundaries (Arabian Sea, Bay of Bengal, 
       and Equatorial Indian Ocean).
    2. Isolates profiles that actively measure Chlorophyll-A ('CHLA').
    3. Groups profiles by unique WMO float IDs and filters out short-lived assets,
       keeping only floats with a lifecycle history of >20 operational cycles.
    4. Maps remaining WMO IDs to their true Data Assembly Center (DAC) server paths.
    5. Downloads missing files over HTTP, skipping any assets already present locally.

    Outputs:
    --------
    None : NoneType
        The function returns no Python objects to the active workspace memory.
    Local Files Created:
    - Directories: Creates folders named 'fixed_[Region_Name]_chl_bioargo' on disk.
    - Files: Saves synthetic profile NetCDF data binary files named '[WMO]_Sprof.nc' 
      directly into their respective regional storage folders.
    """
    print("starting download pipeline...\n")
    
    for region_name, bounds in TARGET_REGIONS.items():
        lat_min, lat_max, lon_min, lon_max = bounds
        
        print(f"processing region: {region_name.replace('_', ' ')}")
        print(f"bounds: lat({lat_min} to {lat_max}), lon({lon_min} to {lon_max})")

        spatial_mask = (index_df['latitude'] >= lat_min) & (index_df['latitude'] <= lat_max) & \
                       (index_df['longitude'] >= lon_min) & (index_df['longitude'] <= lon_max)
        chl_mask = index_df['parameters'].str.contains('CHLA', na=False)
        
        region_df = index_df[spatial_mask & chl_mask].copy()
        
        if region_df.empty:
            print(f"no floats found matching criteria in {region_name}. Skipping.\n")
            continue
            
        region_df['wmo'] = region_df['file'].str.split('/').str[1]

        profile_counts = region_df.groupby('wmo').size()
        frequent_floats = profile_counts[profile_counts > 20].index.tolist()
        
        print(f"found {len(profile_counts)} total floats. filters applied: CHLA present & >20 cycles.")
        print(f"{len(frequent_floats)} floats passed the quality check.")

        download_dir = f"fixed_{region_name}_chl_bioargo"
        os.makedirs(download_dir, exist_ok=True)
        
        wmo_to_dac = dict(zip(region_df['wmo'], region_df['file'].str.split('/').str[0]))

        print(f"downloading NetCDF files to '{download_dir}'...")
        for wmo in sorted(frequent_floats):
            file_name = f"{wmo}_Sprof.nc"
            save_path = os.path.join(download_dir, file_name)

            if os.path.exists(save_path):
                continue

            dac = wmo_to_dac.get(wmo, 'coriolis')
            url = f"{BASE_URL}/{dac}/{wmo}/{file_name}"

            try:
                response = requests.get(url, stream=True, timeout=15)
                if response.status_code == 200:
                    with open(save_path, 'wb') as f:
                        for chunk in response.iter_content(chunk_size=8192):
                            f.write(chunk)
                    print(f"   + saved {file_name}")
                else:
                    print(f"   - failed {wmo} (Status Code: {response.status_code})")
            except Exception as e:
                print(f"error fetching float {wmo}: {e}")
                
        print(f"done processing for {region_name}.\n" + "-"*50 + "\n")

In [ ]:
def extract_float_coordinates(folder_name):
    """
    Extracts spatial positions from local NetCDF files to compute float centroids.

    Inputs:
    -------
    folder_name : str
        The relative or absolute directory path string pointing to a local 
        folder containing unzipped regional BGC-Argo profile files.
        Required file type: Local directory with files ending in '.nc'.

    What it does:
    -------------
    1. Scans the specified local directory for all available NetCDF files.
    2. Parses each file to extract the core WMO ID directly from the filename.
    3. Opens each dataset to calculate the mathematical mean (centroid) of all 
       latitude and longitude trajectory coordinates within that asset file.
    4. Aggregates the records into a single dataset, catching corrupted files.

    Outputs:
    --------
    pandas.DataFrame
        An in-memory DataFrame tracking unique float positions. 
        Structure/Columns:
        - 'wmo' : str, the extracted float identifier.
        - 'latitude' : float, calculated mean deployment latitude.
        - 'longitude' : float, calculated mean deployment longitude.
        Returns an empty DataFrame if no files are found.
    """

    file_list = glob.glob(os.path.join(folder_name, "*.nc"))
    
    if not file_list:
        print(f"no NetCDF files found in '{folder_name}'.")
        return pd.DataFrame()

    data = []
    for file in file_list:
        try:
            with xr.open_dataset(file) as ds:
                # extract WMO from the filename safely across different operating systems
                wmo_id = os.path.basename(file).split('_')[0]
                
                data.append({
                    'wmo': wmo_id,
                    'latitude': ds['LATITUDE'].values.mean(),
                    'longitude': ds['LONGITUDE'].values.mean()
                })
        except Exception as e:
            print(f"could not read {os.path.basename(file)}: {e}")

    return pd.DataFrame(data)

In [ ]:
def plot_float_positions(coords_df, region_name, bounds):
    """
    Plots the calculated float centroids on a Cartopy map to verify spatial positioning.

    Inputs:
    -------
    coords_df : pandas.DataFrame
        In-memory DataFrame containing processed coordinates. Must include 
        columns: 'latitude' and 'longitude'.
    region_name : str
        The identifier string for the ocean target area (e.g., 'Arabian_Sea').
    bounds : tuple of float
        Spatial boundary limits formatted exactly as: (lat_min, lat_max, lon_min, lon_max).

    What it does:
    -------------
    1. Checks if the tracking DataFrame is empty to prevent visualization crashes.
    2. Unpacks the boundary tuple matching the regional dictionary schema layout.
    3. Initializes a geographic Cartopy projection canvas mapping bounding parameters.
    4. Adds high-resolution coastline features, gridlines, and coordinate tick labels.
    5. Projects float coordinate centroids onto the canvas as scatter markers.
    6. Exports a validated static image layout, clipping white margins cleanly.

    Outputs:
    --------
    None : NoneType
        The function returns no Python objects to the active workspace memory.
    Local Files Created:
    - Directory: Ensures a folder named 'plots/' exists on the local drive.
    - File: Saves a standalone image file named 'plots/[region_name]_centroids.png'.
    """
    if coords_df.empty:
        print(f"?? no data to plot for {region_name}")
        return

    lat_min, lat_max, lon_min, lon_max = bounds

    fig = plt.figure(figsize=(10, 7))
    ax = plt.axes(projection=ccrs.PlateCarree())

    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND, facecolor="lightgray", zorder=1)
    ax.add_feature(cfeature.COASTLINE, linewidth=1.5, zorder=2)

    plt.scatter(
        coords_df["longitude"],
        coords_df["latitude"],
        color="darkblue",
        s=50,
        alpha=0.8,
        transform=ccrs.PlateCarree(),
        label=f'{region_name.replace("_", " ")} Floats',
        zorder=3,
        edgecolors="white",
    )

    gl = ax.gridlines(draw_labels=True, linestyle="--", alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False

    clean_title = region_name.replace("_", " ")
    plt.title(f"Argo Float Centroids in the {clean_title}", fontsize=14, pad=15)
    plt.legend(loc="lower left")

    os.makedirs("plots", exist_ok=True)
    output_path = f"plots/{region_name}_centroids.png"

    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"verification plot saved locally to: {output_path}")

In [ ]:
def calculate_regional_climatology(base_dir, region_name):
    """
    Computes vertically interpolated monthly chlorophyll climatologies from profile assets.

    Inputs:
    -------
    base_dir : str
        The relative or absolute directory path string pointing to a local 
        folder containing unzipped regional BGC-Argo profile files.
        Required file type: Local directory with files ending in '.nc'.
    region_name : str
        The identifier string for the ocean target area (e.g., 'Arabian_Sea').

    What it does:
    -------------
    1. Generates a non-uniform standard vertical depth grid (0-200m every 5m, 
       and 250-2000m every 50m).
    2. Opens regional NetCDF files to extract quality-controlled, scientifically 
       adjusted scientific measurements ('CHLA_ADJUSTED', 'PRES_ADJUSTED').
    3. Filters out unphysical values (values <= 0) and records with failing quality 
       control flag markers.
    4. Interpolates individual, uneven vertical profile tracks cleanly onto the uniform 
       depth grid steps.
    5. Groups the independent profiles by calendar month to compute row-wise, 
       statistical mean profiles across the annual matrix.

    Outputs:
    --------
    None : NoneType
        The function returns no Python objects to the active workspace memory.
    Local Files Created:
    - File: Saves a matrix spreadsheet named '[region_name]_Climatology.csv' 
      where rows represent the uniform pressure depth grid (dbar) and the 12 columns 
      represent calendar months (1 to 12).
    """
    print(f"starting processing pipeline for: {region_name}")
    
    std_grid = np.concatenate([np.arange(0, 205, 5), np.arange(250, 2050, 50)])
    monthly_data = {month: [] for month in range(1, 13)}
    
    file_list = glob.glob(os.path.join(base_dir, "*.nc"))
    if not file_list:
        print(f"?? no NetCDF files found in {base_dir}. skipping region.")
        return

    for file_path in file_list:
        try:
            with xr.open_dataset(file_path, decode_times=True) as ds:
                chla = ds['CHLA_ADJUSTED'].values.copy()
                pres = ds['PRES_ADJUSTED'].values
                qc = ds['CHLA_ADJUSTED_QC'].values.astype(str)
                time = ds['JULD'].values

                valid_qc_mask = np.isin(qc, ['1', '2', '5', '8'])
                chla[~valid_qc_mask] = np.nan
                chla[chla <= 0] = np.nan

                for i in range(ds.dims['N_PROF']):
                    month = pd.to_datetime(time[i]).month
                    p_levels = pres[i, :]
                    c_levels = chla[i, :]

                    if np.sum(~np.isnan(c_levels)) > 3:
                        interp_c = np.interp(std_grid, p_levels, c_levels, left=np.nan, right=np.nan)
                        monthly_data[month].append(interp_c)
                        
        except Exception as e:
            print(f"   skipping file {os.path.basename(file_path)} due to error: {e}")

    monthly_means = {}
    for month in range(1, 13):
        data_matrix = monthly_data[month]
        if data_matrix:
            monthly_means[month] = np.nanmean(data_matrix, axis=0)
        else:
            monthly_means[month] = np.zeros(len(std_grid)) * np.nan

    df_climatology = pd.DataFrame(monthly_means, index=std_grid)
    df_climatology.index.name = 'Depth_Pressure_dbar'
    
    output_filename = f"{region_name}_Climatology.csv"
    df_climatology.to_csv(output_filename)
    
    print(f" monthly climatology saved to: {output_filename}\n" + "="*50)

In [ ]:
def calculate_sst_climatology(file_path):
    """
    Extracts and calculates regional monthly Sea Surface Temperature climatologies.

    Inputs:
    -------
    file_path : str
        The relative or absolute system path string pointing to a local multi-decadal 
        global NOAA OISST gridded NetCDF file.
        Required file type: Local binary gridded data file ending in '.nc'.

    What it does:
    -------------
    1. Safeguards execution by checking for file existence to handle missing dependencies.
    2. Opens the comprehensive global dataset using a lazy-loading xarray interface.
    3. Iterates through three predefined spatial boundary sub-configurations (Arabian Sea, 
       Bay of Bengal, and Equatorial Indian Ocean).
    4. Evaluates the latitudinal indexing direction (ascending vs. descending) to safely 
       slice out regional data sub-grids without generating empty inversions.
    5. Collapses spatial dimensions by computing the geographical grid mean (`['lat', 'lon']`).
    6. Groups the remaining 1D time-series data by calendar month and calculates multi-year 
       historical averages for each step.

    Outputs:
    --------
    pandas.DataFrame
        An in-memory matrix tracking regional SST climatology. 
        Structure/Columns:
        - Index: 'Month', string labels spanning 'Jan' through 'Dec'.
        - Columns: 'Arabian_Sea', 'Bay_of_Bengal', and 'Equatorial_IO' containing 
          spatially aggregated monthly mean temperature floats.
        Returns an empty DataFrame if the source file is missing.
    """
    print("loading local NOAA OISST dataset...")
    if not os.path.exists(file_path):
        print(f"error: {file_path} not found. please download.")
        return pd.DataFrame()
        
    ds = xr.open_dataset(file_path)
 
    regions_config = {
        "Arabian_Sea": {"lat": (5, 25), "lon": (50, 75)},
        "Bay_of_Bengal": {"lat": (5, 22), "lon": (80, 100)},
        "Equatorial_IO": {"lat": (-5, 5), "lon": (60, 90)}
    }

    sst_results = {}

    print("Slicing and calculating monthly means across target basins...")
    for name, bounds in regions_config.items():
        lat_min, lat_max = bounds["lat"]
        lon_min, lon_max = bounds["lon"]

        if ds.lat.values[0] > ds.lat.values[-1]:
            lat_slice = slice(lat_max, lat_min)
        else:
            lat_slice = slice(lat_min, lat_max)

        ds_region = ds.sel(lat=lat_slice, lon=slice(lon_min, lon_max))

        spatial_mean = ds_region['sst'].mean(dim=['lat', 'lon'])
        monthly_clim = spatial_mean.groupby('time.month').mean('time').values

        sst_results[name] = monthly_clim

    months_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    df_sst = pd.DataFrame(sst_results, index=months_labels)
    df_sst.index.name = 'Month'
    
    return df_sst

In [ ]:
def plot_final_climatology_grid(df_sst):
    """
    Generates a 12x3 subplot matrix pairing vertical Chlorophyll-A profiles with SST values.

    Inputs:
    -------
    df_sst : pandas.DataFrame
        In-memory matrix containing regional Sea Surface Temperature climatology data.
        Required columns: 'Arabian Sea', 'Bay of Bengal', and 'Equatorial IO'.
        Required index: Named monthly string identifiers ('Jan' through 'Dec').

    What it does:
    -------------
    1. Reads previously generated regional climatology CSV files and safely maps 
       string index rows to numeric depths.
    2. Scans the upper 300 meters of all regional datasets to find a global peak 
       Chlorophyll value, standardizing the X-axis limit dynamically for clean comparison.
    3. Initializes a massive 12-row by 3-column matplotlib subplot canvas representing 
       calendar months and target ocean basins.
    4. Plots vertical profile tracks (depth vs concentration) and adds an automated 
       horizontal dashed line marking the Deep Chlorophyll Maximum (DCM) depth.
    5. Inverts the Y-axis (`300` to `0`) to properly simulate looking down the water column.
    6. Overlay a text bounding box containing the corresponding NOAA monthly mean SST value 
       in the corner of every subplot panel.

    Outputs:
    --------
    None : NoneType
        The function returns no Python objects to the active workspace memory.
    Local Files Created:
    - Directory: Ensures a folder named 'plots/' exists on the local drive.
    - File: Saves a comprehensive high-resolution composite dashboard image file 
      named 'plots/Master_Climatology_Profiles.png'.
    """
    print("constructing the 12x3 Chlorophyll-SST matrix plot...")
    
    regions_files = {
        "Arabian Sea": "Arabian_Sea_Climatology.csv",
        "Bay of Bengal": "Bay_of_Bengal_Climatology.csv",
        "Equatorial IO": "Equatorial_IO_Climatology.csv"
    }

    months_str = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    global_chl_max = 0.0
    loaded_dfs = {}

    for region_name, filename in regions_files.items():
        if os.path.exists(filename):
            df = pd.read_csv(filename, index_col=0)
            df.index = pd.to_numeric(df.index)
            loaded_dfs[region_name] = df

            upper_300m_df = df.loc[df.index <= 300]
            if not upper_300m_df.empty:
                max_val = upper_300m_df.max().max()
                if max_val > global_chl_max:
                    global_chl_max = max_val

    x_max_limit = global_chl_max * 1.1 if global_chl_max > 0 else 2.0

    fig, axes = plt.subplots(12, 3, figsize=(18, 40), constrained_layout=True)

    for row_idx, month_name in enumerate(months_str):
        col_name = str(row_idx + 1)  # Months columns are labeled '1' through '12'

        for col_idx, (region_name, _) in enumerate(regions_files.items()):
            ax = axes[row_idx, col_idx]

            if region_name in loaded_dfs:
                df = loaded_dfs[region_name]

                if col_name in df.columns:
                    data = df[[col_name]].dropna()
                    x = data[col_name].values
                    y = data.index.values

                    ax.plot(x, y, color='darkgreen', linewidth=1.5)

                    if len(x) > 0:
                        max_idx = np.argmax(x)
                        dcm_depth = y[max_idx]
                        plot_depth = 5 if dcm_depth == 0 else dcm_depth
                        ax.axhline(y=plot_depth, color='darkgreen', linestyle='--', alpha=0.8, linewidth=1.5)

            ax.set_ylim(300, 0) # Focus on upper 300 meters
            ax.set_xlim(0, x_max_limit)

            ax.xaxis.set_ticks_position('top')
            ax.xaxis.set_label_position('top')
            ax.set_xlabel('Chl-a (mg/m³)')
            ax.grid(True, linestyle=':', alpha=0.5)

            if col_idx == 0:
                ax.set_ylabel('Depth (m)')
                ax.text(-0.35, 0.5, month_name, transform=ax.transAxes,
                        fontsize=12, fontweight='bold', rotation=90, va='center')

            if region_name in df_sst.columns:
                sst_val = df_sst.loc[month_name, region_name]
                ax.text(0.95, 0.95, f'SST: {sst_val:.1f}°C',
                        transform=ax.transAxes, fontsize=10, fontweight='bold',
                        ha='right', va='top', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

            if row_idx == 0:
                ax.set_title(region_name, fontsize=14, fontweight='bold', pad=30)

    os.makedirs("plots", exist_ok=True)
    output_img_path = "plots/Master_Climatology_Profiles.png"
    plt.savefig(output_img_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"profile matrix saved to: {output_img_path}")

In [ ]:

if __name__ == "__main__":
    
    print(" Fetching the global Argo bio-profile index...")
    index_url = "https://data-argo.ifremer.fr/argo_bio-profile_index.txt.gz"
    global_df = pd.read_csv(index_url, skiprows=8)
    
    download_bioargo_data(global_df)

    for region, map_limits in PLOT_BOUNDS.items():
        folder_path = f"fixed_{region}_chl_bioargo"
        df_coords = extract_float_coordinates(folder_path)
        plot_float_positions(df_coords, region, map_limits)
    print(" local coordinate checks and maps generated inside 'plots/' folder.")

    for region in TARGET_REGIONS.keys():
        calculate_regional_climatology(f"fixed_{region}_chl_bioargo", region)

    noaa_url = "https://downloads.psl.noaa.gov/Datasets/noaa.oisst.v2/sst.mnmean.nc"
    local_sst_file = "sst.mnmean.nc"
    if not os.path.exists(local_sst_file):
        print("local NOAA OISST file missing. starting download...")
        urllib.request.urlretrieve(noaa_url, local_sst_file)
    
    df_sst_final = calculate_sst_climatology(local_sst_file)
    df_sst_final.to_csv("regional_SST_climatology.csv")

    df_sst_plotting = df_sst_final.rename(columns={
        "Arabian_Sea": "Arabian Sea",
        "Bay_of_Bengal": "Bay of Bengal",
        "Equatorial_IO": "Equatorial IO"
    })
    plot_final_climatology_grid(df_sst_plotting)
    
    print("\ndonedonedone")f